## Overview

This notebook loads the pickled results produced by `scaling_effdim_bonddim.py` and plots the normalized effective dimension against the tensor-train bond dimension `chi` used to represent V (the parameter-side singular-vector matrix in the SVD Gamma = U @ diag(S) @ V^T of the structure constants), for several combinations of number of parameters M, local basis dimension Dloc (d_tilde), and input dimension D. Each curve traces the Monte-Carlo estimate of the normalized effective dimension (Eq. 12 of the paper) as bond dimension varies, annotated with the corresponding correlation-spectrum purity tr(S^4).

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import tensor_network_functions_jax
importlib.reload(tensor_network_functions_jax)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the fixed scan parameters (number of parameter samples, number of random V realizations per bond dimension, decay exponent of the correlation spectrum S) and lists the (M, Dloc, D) combinations whose pickled bond-dimension scans (produced by `scaling_effdim_bonddim.py`) will be loaded and overlaid.

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_bonddim_Vh/'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 200
no_par_samples_name = str(no_samples)

# No. of random TT realizations per bond dim
no_matrix_realiz = 20
no_V_samples_name = str(no_matrix_realiz)

# Decay exponent
dec_exp = 0.0
dec_exp_name = '0p0'

########################### Exps. to plot ###########################

# No. of parameters
no_params_vec = [20, 40, 30, 30, 30, 40]
name_no_params_vec = [str(a) for a in no_params_vec]

# Local dimension of parameter functions space
local_dim_param_vec = [3, 3, 5, 5, 7, 7]
name_dim_par_loc_vec = [str(a) for a in local_dim_param_vec]

# Input space dimension
dim_in_vec = [20, 20, 20, 60, 60, 20]
name_dim_in_vec = [str(a) for a in dim_in_vec]

no_exps = len(no_params_vec)

For each (M, Dloc, D) combination, loads and concatenates the pickled bond dimension, correlation-spectrum purity tr(S^4), and normalized effective dimension arrays across all matching result files into `dict_exps`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    no_params = no_params_vec[i]
    local_dim_param = local_dim_param_vec[i]
    name_no_params = name_no_params_vec[i]
    name_dim_par_loc = name_dim_par_loc_vec[i]
    dim_in = dim_in_vec[i]
    name_dim_in = name_dim_in_vec[i]

    name_end = ('_DimIn' + name_dim_in + '_DimLocPar' + name_dim_par_loc + 
                '_Nparams' + name_no_params + '_NsamplesPar' + no_par_samples_name + 
                '_NsamplesV' + no_V_samples_name + '_DecExp' + dec_exp_name + '.pkl')
    
    namefileini = 'norm_eff_dim'
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini) and
                                                              f.endswith(name_end))]
    print('M=' + name_no_params + ', D=' + name_dim_in + ', dloc=' + name_dim_par_loc + ' : ', len(listallfiles))
    
    bonddim_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        bonddim_vec = dict_norm_ed['bonddim_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    
        bonddim_all.append(bonddim_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    bonddim_all = np.concatenate(bonddim_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['no_params'] = no_params
    dict_exp['local_dim_param'] = local_dim_param
    dict_exp['dim_in'] = dim_in
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['bonddim_all'] = bonddim_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Plots the normalized effective dimension d_eff vs. bond dimension chi (log-scaled x-axis) for selected configurations, labeling each curve with Dloc (d_tilde), M, and the correlation-spectrum purity tr(S^4).

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

#clrs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
#for i in range(no_exps):
clrs = ['tab:blue', 'tab:orange', 'tab:orange', 'tab:green', 'tab:green', 'tab:green']
for i in [0, 2, 5]:
#clrs = ['tab:blue', 'tab:orange', 'tab:orange', 'tab:blue', 'tab:orange', 'tab:green']
#for i in [3, 4]:
    dict_exp = dict_exps[i]
    dim_in = dict_exp['dim_in']
    bonddim_all = dict_exp['bonddim_all']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    no_params = dict_exp['no_params']
    local_dim_param = dict_exp['local_dim_param']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    purity_rounded = round(numpy.mean(purity_S_all), 2)
    print('S purity: ', purity_rounded, ' with std. ', np.std(purity_S_all))
    #lbl = r'$\tilde{d}=' + str(local_dim_param) + ',\,M=' + str(no_params) + ',\,D=' + str(dim_in) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    lbl = r'$\tilde{d}=' + str(local_dim_param) + ',\,M=' + str(no_params) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    x = np.asarray(bonddim_all)
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[i], markersize=10, label=lbl)

ax.legend(fontsize=22)
ax.set_xlabel(r'$\chi$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_ylim([0.3, 1.0])

fig.savefig('EffDimScaling_vs_BondDimV_TN__D_20.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_BondDimV_TN__D_20.pdf', bbox_inches='tight')
#fig.savefig('EffDimScaling_vs_BondDimV_TN__D_60.png', bbox_inches='tight')
#fig.savefig('EffDimScaling_vs_BondDimV_TN__D_60.pdf', bbox_inches='tight')